In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

In [2]:
# 1. Synthesize normal operational data (e.g., Server Memory vs CPU Usage)
np.random.seed(42)
normal_data = np.random.normal(loc=50, scale=5, size=(200, 2))

# Inject 5 highly visible malicious anomalies/outliers
anomalies = np.array([[15, 85], [80, 12], [90, 95], [10, 5], [5, 90]])
X = np.vstack([normal_data, anomalies])

In [3]:
# 2. Standardize features (Mandatory for One-Class SVM geometric calculation)
X_scaled = StandardScaler().fit_transform(X)

In [4]:
# 3. Initialize and fit the Isolation Forest
# contamination=0.025 tells the model we expect roughly 2.5% anomalies
iso_forest = IsolationForest(contamination=0.025, random_state=42)
iso_preds = iso_forest.fit_predict(X_scaled) 
# Note: scikit-learn anomaly detectors output 1 for Inliers, -1 for Outliers

In [5]:
# 4. Initialize and fit the One-Class SVM
# nu matches our contamination expectation; gamma sets the boundary smoothness
oc_svm = OneClassSVM(nu=0.025, kernel='rbf', gamma=0.1)
svm_preds = oc_svm.fit_predict(X_scaled)

In [6]:
# 5. Compile results for analysis (evaluating the last 5 injected anomaly rows)
results = pd.DataFrame({
    'CPU_Usage': X[-5:, 0],
    'Memory_Usage': X[-5:, 1],
    'IsoForest_Detection': iso_preds[-5:],
    'OneClassSVM_Detection': svm_preds[-5:]
})

# Remap -1 to "Anomaly" and 1 to "Normal" for production readability
results.replace({1: 'Normal', -1: 'Anomaly'}, inplace=True)

print("--- Anomaly Detection Audit (Last 5 Flagged Rows) ---")
print(results)

--- Anomaly Detection Audit (Last 5 Flagged Rows) ---
   CPU_Usage  Memory_Usage IsoForest_Detection OneClassSVM_Detection
0       15.0          85.0             Anomaly                Normal
1       80.0          12.0             Anomaly                Normal
2       90.0          95.0             Anomaly               Anomaly
3       10.0           5.0             Anomaly               Anomaly
4        5.0          90.0             Anomaly                Normal
